# Phase 5: Parabolic IDSM — Moving Inhomogeneities

**Project**: Demystifying Iterative Direct Sampling Methods — From Theory to Code  
**Reference**: Jin, Wang, and Zou, *An Iterative Direct Sampling Method for Reconstructing Moving Inhomogeneities in Parabolic Problems* (arXiv:2511.08197)  
**Objective**: Reproduce the paper's Section 5 parabolic experiments from the original FreeFEM programs, while keeping the derivations, code comments, and figure outputs consistent with Notebooks 01–04.

---

Phase 5 extends IDSM from static elliptic inclusions to time-dependent parabolic inverse problems. On the unit disk
$$
\Omega = \{x \in \mathbb{R}^2 : |x| < 1\},
$$
the linear examples use the parabolic initial-boundary value problem
$$
\begin{aligned}
\partial_t y - \nabla\!\cdot(\sigma(x,t)\nabla y) + V(x,t)y &= F &&\text{in } \Omega\times(0,T),\\
\sigma\partial_\nu y &= f &&\text{on } \partial\Omega\times(0,T),\\
y(\cdot,0) &= y_0 &&\text{in } \Omega.
\end{aligned}
$$
The measured lateral Cauchy pair is $(f,y^d)(t)$; the synthetic data use the same multiplicative boundary-value noise convention as Phase 1,
$$
y^d_h(t_i,x_j)=y_h(t_i,x_j)+\varepsilon\xi_{ij}|y_h(t_i,x_j)|,\qquad \xi_{ij}\in[-1,1].
$$
Example 5.3 replaces the linear zeroth-order term by the nonlinear source $|y|y\,U$. Algorithm 4.1 reconstructs the coefficient on each segment $[t_k,t_{k+1}]$ as the unknown inhomogeneity moves, merges, fades, or shrinks over time.

This notebook follows the same pattern as the first four notebooks:

1. state the mathematical object being discretized,
2. run the corresponding Python implementation against the FreeFEM configuration,
3. save each figure under `../figures/05_*.png`, and
4. summarize only values computed by the live cells above.

The five examples cover conductivity-only, mixed conductivity/potential, nonlinear $|y|y\,U$, potential fading, and conductivity diminishing cases.

In [ ]:
import os
import sys
import time
from pathlib import Path

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
from matplotlib.patches import Ellipse

from src.mesh import generate_disk_mesh, generate_disk_mesh_paper
from src.idsm_parabolic import (
    run_idsm_parabolic,
    edp_cfg_example_5_1, paper_cfg_example_5_1,
    edp_cfg_example_5_2, paper_cfg_example_5_2,
    edp_cfg_example_5_3, paper_cfg_example_5_3,
    edp_cfg_example_5_4, paper_cfg_example_5_4,
    edp_cfg_example_5_5, paper_cfg_example_5_5,
    trajectory_example_5_1, radius_example_5_1,
    trajectory_example_5_2,
    trajectory_example_5_3,
    trajectory_example_5_4, radius_example_5_4,
    trajectory_example_5_5, radius_example_5_5,
    c_func_example_5_1, v_func_example_5_1,
    c_func_example_5_2, v_func_example_5_2,
    c_func_example_5_3, v_func_example_5_3,
    c_func_example_5_4, v_func_example_5_4,
    c_func_example_5_5, v_func_example_5_5,
    u_func_example_5_3,
    synthesize_full_forward,
)

FIG_DIR = Path('../figures')
FIG_DIR.mkdir(exist_ok=True)


def save_figure(fig, filename):
    """Save a notebook figure using the same convention as NB01-NB04."""
    path = FIG_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  saved {path}')
    return path


## 1. Paper §5 Setup — FreeFEM Three-Mesh Architecture

The original FreeFEM programs use three distinct meshes:

- `ThFine`: fine P1 mesh for synthesizing noisy forward data.
- `Th`: P1 solve mesh for empty, adjoint, inhomogeneous, and Dirichlet verification PDE solves.
- `ThCoarse`: P0 coefficient mesh for local dual projection, low-rank resolver `R_k`, and stored reconstruction histories.

The strict Python driver `scripts/run_all_examples.py --mesh-mode edp` mirrors this layout with `data_mesh / solve_mesh / coeff_mesh` generated from each example's `nSolve` and `nCoarse`. The notebook keeps a paper-scale interactive triplet (`13870 / 7002 / 1120` target triangles) so the code path is the same as the script, while the full reproducible runs should be launched from the `.py` driver.

The time grid follows the reference parameters:

- forward data step `forward_dt` (paper text: $\Delta t = 0.01$; individual `.edp` files may use `0.015` or `0.02`),
- inverse segment length `delta_t = 0.1`,
- substeps per inverse segment `delta_t_split`, and
- the FreeFEM loop convention `tIndex < floor(totalTime / deltaT) - 1`.


In [ ]:
data_mesh = generate_disk_mesh_paper(target_triangles=13870)
solve_mesh = generate_disk_mesh_paper(target_triangles=7002)
coeff_mesh = generate_disk_mesh_paper(target_triangles=1120)

# Backward-compatible aliases used by older cells in this notebook.
fine_mesh = data_mesh
coarse_mesh = coeff_mesh

print(f'data  (ThFine) : {data_mesh.triangles.shape[0]} tri, {data_mesh.points.shape[0]} nodes')
print(f'solve (Th)     : {solve_mesh.triangles.shape[0]} tri, {solve_mesh.points.shape[0]} nodes')
print(f'coeff (ThCoarse): {coeff_mesh.triangles.shape[0]} tri, {coeff_mesh.points.shape[0]} nodes')

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
for ax, m, ttl in zip(
    axes,
    [data_mesh, solve_mesh, coeff_mesh],
    ['ThFine data (≈13870)', 'Th solve (≈7002)', 'ThCoarse coeff (≈1120)'],
):
    triang = Triangulation(m.points[:, 0], m.points[:, 1], m.triangles)
    ax.triplot(triang, color='steelblue', linewidth=0.2)
    ax.set_title(f'{ttl}: {m.triangles.shape[0]} tri')
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
save_figure(fig, '05_disk_meshes.png')
plt.show()


## 2. Forward Crank–Nicolson Sanity Check

`synthesize_full_forward` performs a full P1 Crank–Nicolson time-stepping sweep on ThFine (matching `.edp` L196-253). We run it once on the Ex 5.1 noiseless cfg to check:

- `y_clean[0] = initial_data`
- `y_clean[-1]` has a sensible spatial structure on the boundary, jointly modulated by the time-dependent `BdSource` and the inclusion positions.

In [ ]:
cfg_smoke = edp_cfg_example_5_1(noise=0.0)
cfg_smoke.total_time = 0.5  # 5 substep x 0.1 sanity check only

y_data, y_clean = synthesize_full_forward(
    data_mesh, cfg_smoke,
    c_func_example_5_1, v_func_example_5_1,
    rng=np.random.default_rng(42),
)
print(f'y_clean shape: {y_clean.shape}  (n_steps, data_num, n_pts_data)')
print(f'  t=0    range : [{y_clean[0, 0].min():.3f}, {y_clean[0, 0].max():.3f}]')
print(f'  t=0.5  range : [{y_clean[-1, 0].min():.3f}, {y_clean[-1, 0].max():.3f}]')

triang = Triangulation(data_mesh.points[:, 0], data_mesh.points[:, 1], data_mesh.triangles)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
n_step = y_clean.shape[0] - 1
for ax, k in zip(axes, [0, n_step // 3, 2 * n_step // 3, n_step]):
    im = ax.tripcolor(triang, y_clean[k, 0], shading='gouraud', cmap='RdBu_r')
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f't={k * cfg_smoke.forward_dt:.3f}')
    fig.colorbar(im, ax=ax, fraction=0.045)
fig.suptitle('CN forward sanity check (Ex 5.1 noiseless, T=0.5)', y=1.02)
fig.tight_layout()
save_figure(fig, '05_forward_cn_sanity.png')
plt.show()

## 3. Helper — Live runner & heatmap

`run_example` calls `run_idsm_parabolic` directly and returns an in-memory dict with:

- `sigma_history` $(n_{\rm seg}, n_{\rm tri\, coarse})$ — reconstructed conductivity P0
- `v_history` $(n_{\rm seg}, n_{\rm tri\, coarse})$ — reconstructed potential P0
- `iou_history` $(n_{\rm seg},)$ — IoU at the end of each segment
- `n_inner_per_segment`, `residuals_per_segment`
- `coarse_points`, `coarse_triangles` — reused for plotting

The `heatmap_row` and `iou_curve` helpers below take this dict and overlay the inclusion ellipses given by `trajectory_example_5_X(t, traj_index)` + `radius_*`.


In [ ]:
GHOST_R = 1e-6  # FreeFEM uses radius 1e-10 to mark inactive ghost inclusions.


def _slug(text):
    """Return a compact ASCII filename component."""
    out = []
    for ch in text.lower():
        out.append(ch if (ch.isascii() and ch.isalnum()) else '_')
    return '_'.join(''.join(out).split('_')).strip('_')


def run_example(name, cfg, coarse_mesh, fine_mesh, c_func, v_func, *, seed=42, solve_mesh=None):
    """Run one parabolic IDSM example and normalize the result dictionary."""
    expected_segments = cfg.n_segments
    solve_mesh = globals().get('solve_mesh', coarse_mesh) if solve_mesh is None else solve_mesh
    print(f'  [{name}] running: total_time={cfg.total_time:.2f}, n_seg={expected_segments}, '
          f'data/solve/coeff tri={fine_mesh.n_triangles}/{solve_mesh.n_triangles}/{coarse_mesh.n_triangles}')
    t0 = time.perf_counter()
    res = run_idsm_parabolic(coarse_mesh, fine_mesh, cfg, c_func, v_func, seed=seed, solve_mesh=solve_mesh)
    dt = time.perf_counter() - t0
    iou = np.asarray(res['iou_history'])
    print(f'  [{name}] done: runtime={dt:.1f}s, IoU mean={iou.mean():.3f}, '
          f'max={iou.max():.3f} @seg{int(iou.argmax())}')
    return {
        'sigma_history': np.asarray(res['sigma_history']),
        'v_history': np.asarray(res['v_history']),
        'iou_history': iou,
        'n_inner_per_segment': np.asarray(res['n_inner_per_segment']),
        'residuals_per_segment': res['residuals_per_segment'],
        'coarse_points': coarse_mesh.points,
        'coarse_triangles': coarse_mesh.triangles,
        'cfg_total_time': cfg.total_time,
        'cfg_cA': cfg.cA,
        'cfg_cB': cfg.cB,
        'cfg_vA': cfg.vA,
        'cfg_vB': cfg.vB,
        'runtime_seconds': dt,
        'name': name,
        'slug': _slug(name),
    }


def _segment_times(rec):
    """FreeFEM segment end times: t_k = (k + 1) deltaT."""
    n_seg = rec['iou_history'].shape[0]
    return np.arange(1, n_seg + 1) * (float(rec['cfg_total_time']) / n_seg)


def _draw_inclusions(ax, traj_func, radius_func, t_now, idx_tuple, edge='red'):
    """Draw ground-truth inclusion outlines and skip inactive ghost entries."""
    for ti in idx_tuple:
        cp = traj_func(t_now, ti)
        r = radius_func(t_now, ti)
        if max(float(r[0]), float(r[1])) < GHOST_R:
            continue
        ax.add_patch(Ellipse(
            (float(cp[0]), float(cp[1])),
            width=2.0 * float(r[0]),
            height=2.0 * float(r[1]),
            fill=False,
            edgecolor=edge,
            linewidth=1.5,
        ))


def heatmap_row(rec, traj_func, radius_func, idx_tuple, *,
                model='cond', vlow=None, vhigh=None, cmap='viridis',
                n_frames=6, ylabel='sigma', edge='red'):
    """Plot selected segment reconstructions; the caller saves the figure."""
    sigma_h = rec['sigma_history']
    v_h = rec['v_history']
    n_seg = sigma_h.shape[0]
    seg_t = _segment_times(rec)
    pts = rec['coarse_points']
    tris = rec['coarse_triangles']
    triang = Triangulation(pts[:, 0], pts[:, 1], tris)

    n_frames = min(n_frames, n_seg)
    idxs = np.linspace(0, n_seg - 1, n_frames).astype(int)

    fig, axes = plt.subplots(1, n_frames, figsize=(2.4 * n_frames, 2.7))
    if n_frames == 1:
        axes = [axes]
    for ax, k in zip(axes, idxs):
        field = sigma_h[k] if model == 'cond' else v_h[k]
        ax.tripcolor(triang, facecolors=field, shading='flat',
                     cmap=cmap, vmin=vlow, vmax=vhigh)
        _draw_inclusions(ax, traj_func, radius_func, float(seg_t[k]),
                         idx_tuple, edge=edge)
        ax.set_xlim(-1.05, 1.05)
        ax.set_ylim(-1.05, 1.05)
        ax.set_aspect('equal')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f't={seg_t[k]:.2f}')
    axes[0].set_ylabel(ylabel, fontsize=12)
    return fig


def iou_curve(rec_list, labels):
    """Plot IoU histories for one or more records; the caller saves the figure."""
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for rec, lbl in zip(rec_list, labels):
        ax.plot(_segment_times(rec), rec['iou_history'], label=lbl, linewidth=1.5)
    ax.set_xlabel('t')
    ax.set_ylabel('IoU(t)')
    ymax = max(0.4, max(np.asarray(r['iou_history']).max() for r in rec_list) * 1.1)
    ax.set_ylim(0, ymax)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)
    return fig


# Radius helpers for examples whose radii are constants in the FreeFEM code.
def radius_5_2(t, i):
    return np.array([0.2, 0.2]) if i in (0, 1, 2) else np.array([1e-10, 1e-10])


def radius_5_3(t, i):
    return np.array([0.2, 0.2]) if i == 0 else np.array([1e-10, 1e-10])


## 4. Example 5.1 — Conductivity Merging (paper §5.1)

Two conductivity inclusions ($D_1, D_2$) move together for $t<3$, merge for $3\leq t<6$, and split again for $t\geq6$, each with radius $0.2$. Configuration `paper`: $\varepsilon=5\%$ with the FreeFEM defaults `tol=0.08`, `max_inner=80`, `forget=0.7`.

In [ ]:
rec_5_1 = run_example('5.1 paper ε=5%',
                       paper_cfg_example_5_1(noise=0.05),
                       coarse_mesh, fine_mesh,
                       c_func_example_5_1, v_func_example_5_1)

cA = rec_5_1['cfg_cA']; cB = rec_5_1['cfg_cB']
fig = heatmap_row(rec_5_1,
                  lambda t, i: trajectory_example_5_1(t, i),
                  lambda t, i: radius_example_5_1(i),
                  idx_tuple=(0, 1),
                  model='cond', vlow=cB, vhigh=cA, cmap='viridis', n_frames=6)
fig.suptitle('Ex 5.1 ConductivityMerging — paper configuration', fontsize=11)
fig.tight_layout()
save_figure(fig, '05_ex5_1_conductivity_merging.png')
plt.show()


### 4.1 Noise robustness — $\varepsilon = 5\%$ vs $10\%$

`scripts/run_all_examples.py` with `--include-ex51-n10` runs the paper configuration at noise=10% in addition to the default 5%.

In [ ]:
rec_5_1_n10 = run_example('5.1 paper ε=10%',
                            paper_cfg_example_5_1(noise=0.10),
                            coarse_mesh, fine_mesh,
                            c_func_example_5_1, v_func_example_5_1)

fig = iou_curve([rec_5_1, rec_5_1_n10], ['ε=5%', 'ε=10%'])
fig.suptitle('Ex 5.1 — IoU(t) under boundary noise', fontsize=11)
fig.tight_layout()
save_figure(fig, '05_ex5_1_noise_iou.png')
plt.show()


## 5. Example 5.2 — Mixed Moving (σ + V double model, paper §5.2)

Two σ inclusions (traj 0/1) plus one V inclusion (traj 2), all with plain Euclidean radius $0.2$. `model='double'` recovers σ and V simultaneously; `vB=15.0`, `lowrank='DFP'`.

In [ ]:
rec_5_2 = run_example('5.2 paper',
                       paper_cfg_example_5_2(noise=0.05),
                       coarse_mesh, fine_mesh,
                       c_func_example_5_2, v_func_example_5_2)

cA = rec_5_2['cfg_cA']; cB = rec_5_2['cfg_cB']
vA = rec_5_2['cfg_vA']; vB = rec_5_2['cfg_vB']

fig_s = heatmap_row(rec_5_2,
                    lambda t, i: trajectory_example_5_2(t, i),
                    radius_5_2, idx_tuple=(0, 1),
                    model='cond', vlow=cB, vhigh=cA,
                    cmap='viridis', n_frames=6, ylabel='σ', edge='red')
fig_s.suptitle('Ex 5.2 MixedMoving — sigma reconstruction', fontsize=11)
fig_s.tight_layout()
save_figure(fig_s, '05_ex5_2_mixed_sigma.png')
plt.show()

fig_v = heatmap_row(rec_5_2,
                    lambda t, i: trajectory_example_5_2(t, i),
                    radius_5_2, idx_tuple=(2,),
                    model='pot', vlow=vA, vhigh=max(vB, vA + 1e-6),
                    cmap='magma', n_frames=6, ylabel='V', edge='cyan')
fig_v.suptitle('Ex 5.2 MixedMoving — potential reconstruction', fontsize=11)
fig_v.tight_layout()
save_figure(fig_v, '05_ex5_2_mixed_potential.png')
plt.show()


## 6. Example 5.3 — Nonlinear ($p=3$, paper §5.3)

The forward problem contains a cubic source term $|y| y \cdot U$. The solver `solve_forward_segment_nonlinear` performs Newton + Crank–Nicolson iterations matching the nonlinear FreeFEM reference; the inverse side runs the dedicated `iterate_segment_nonlinear` / `finalize_segment_nonlinear` branch. Its gradient uses
$$
\zeta_U = \frac12\big(|y_g|y_g + |y_L|y_L\big)y_{\rm dual}\,\mathrm{normalScale},
$$
followed by the box projection $u \in [u_A, 2u_B]$ on the coarse P0 mesh.

This is **not** a σ/V double-field reconstruction: only the U-coefficient field is recovered. The cfg flag `cfg.model='nonlinear'` dispatches this path inside `run_idsm_parabolic`.

We run the paper §5.3 long-time scenario with `total_time=10.0` / about 100 inverse segments. The FreeFEM script's default `totalTime=0.51` is a short debug run; extending it is necessary for the moving inclusion to separate visibly. `paper_cfg_example_5_3` keeps the same numerical defaults as `edp_cfg_example_5_3` and differs only in the paper-style noise setting supplied by the caller.


In [ ]:
# Ex 5.3 — Nonlinear, paper-scale: T=10.0 / about 100 segments.
# The .edp default totalTime=0.51 is a short debug horizon, so we manually
# extend it to 10.0 for the long-time paper-style visualization.
def _cfg_5_3(mode):
    cfg = paper_cfg_example_5_3(noise=0.05) if mode == 'paper' else edp_cfg_example_5_3(noise=0.05)
    cfg.total_time = 10.0  # long-time paper-scale run
    return cfg

rec_5_3_paper = run_example('5.3 paper T=10', _cfg_5_3('paper'),
                             coarse_mesh, fine_mesh,
                             c_func_example_5_3, u_func_example_5_3)
rec_5_3_edp   = run_example('5.3 edp   T=10', _cfg_5_3('edp'),
                             coarse_mesh, fine_mesh,
                             c_func_example_5_3, u_func_example_5_3)

# Heatmap row for the edp configuration, which keeps the FreeFEM inner-loop defaults.
cA = rec_5_3_edp['cfg_cA']; cB = rec_5_3_edp['cfg_cB']
fig = heatmap_row(rec_5_3_edp,
                  lambda t, i: trajectory_example_5_3(t, i),
                  radius_5_3, idx_tuple=(0,),
                  model='cond', vlow=cB, vhigh=cA + 1e-3,
                  cmap='viridis', n_frames=6)
fig.suptitle('Ex 5.3 Nonlinear (p=3, edp cfg, T=10) — U recovery', fontsize=11)
fig.tight_layout()
save_figure(fig, '05_ex5_3_nonlinear_u_recovery.png')
plt.show()

fig2 = iou_curve([rec_5_3_paper, rec_5_3_edp], ['paper', 'edp'])
fig2.suptitle('Ex 5.3 — IoU(t), paper vs edp configurations', fontsize=11)
fig2.tight_layout()
save_figure(fig2, '05_ex5_3_paper_vs_edp_iou.png')
plt.show()


## 7. Example 5.4 — Potential Fading (paper §5.4)

$\sigma \equiv c_A$ is known, so only V is recovered. Two V inclusions:

- traj 2: $V_1(t) = \max(v_B + t(v_A-v_B)/6, v_A)$ — fades from $v_B=15$ to $v_A$ over 6 seconds
- traj 3: $V_2(t) = \min(v_A + t(v_B-v_A)/6, v_B)$ — grows back in the reverse direction

`model='potential'`, `cB = cA + 1e-10` (deliberately degenerate σ).

In [ ]:
rec_5_4 = run_example('5.4 paper',
                       paper_cfg_example_5_4(noise=0.05),
                       coarse_mesh, fine_mesh,
                       c_func_example_5_4, v_func_example_5_4)

vA = rec_5_4['cfg_vA']; vB = rec_5_4['cfg_vB']
fig = heatmap_row(rec_5_4,
                  lambda t, i: trajectory_example_5_4(t, i),
                  lambda t, i: radius_example_5_4(i),
                  idx_tuple=(2, 3),
                  model='pot', vlow=vA, vhigh=max(vB, vA + 1e-6),
                  cmap='magma', n_frames=6, ylabel='V', edge='cyan')
fig.suptitle('Ex 5.4 PotentialFading — potential reconstruction', fontsize=11)
fig.tight_layout()
save_figure(fig, '05_ex5_4_potential_fading.png')
plt.show()

fig2, ax = plt.subplots(figsize=(7, 3.2))
n4 = rec_5_4['iou_history'].shape[0]
T4 = float(rec_5_4['cfg_total_time'])
t4 = np.arange(1, n4 + 1) * (T4 / n4)
ax.plot(t4, rec_5_4['iou_history'])
ax.set_xlabel('t'); ax.set_ylabel('IoU(t)')
ax.set_title('Ex 5.4 — IoU(t) (V channel)'); ax.grid(alpha=0.3)
fig2.tight_layout()
save_figure(fig2, '05_ex5_4_potential_iou.png')
plt.show()


## 8. Example 5.5 — Conductivity Diminishing (paper §5.5)

Two σ inclusions, with traj 1 shrinking over time: $r(t) = \max(0.3 - 0.03t, 10^{-10})$ — it vanishes after 10 seconds. `total_time=5.31`, `nSolve=100`, `lowrank='DFP'`.

In [ ]:
rec_5_5 = run_example('5.5 paper',
                       paper_cfg_example_5_5(noise=0.05),
                       coarse_mesh, fine_mesh,
                       c_func_example_5_5, v_func_example_5_5)

cA = rec_5_5['cfg_cA']; cB = rec_5_5['cfg_cB']
fig = heatmap_row(rec_5_5,
                  lambda t, i: trajectory_example_5_5(t, i),
                  lambda t, i: radius_example_5_5(t, i),
                  idx_tuple=(0, 1),
                  model='cond', vlow=cB, vhigh=cA,
                  cmap='viridis', n_frames=6)
fig.suptitle('Ex 5.5 ConductivityDiminishing — conductivity reconstruction', fontsize=11)
fig.tight_layout()
save_figure(fig, '05_ex5_5_conductivity_diminishing.png')
plt.show()


## 9. Live Summary — Iterations, Solve Counts, and IoU

The paper's Table 1 reports average PDE solves per time segment. The Python driver records the inner-loop count per segment; the estimate below uses the same accounting structure as the FreeFEM loop: each inner iteration adds one inhomogeneous forward solve and one auxiliary adjoint solve, while each segment also has the background solve, the first adjoint solve, and the final Dirichlet-corrected forward solve.

The table is generated from the live records above so it stays synchronized with a fresh notebook run.

In [ ]:
records = [
    ('5.1 paper eps=5%', rec_5_1),
    ('5.1 paper eps=10%', rec_5_1_n10),
    ('5.2 paper', rec_5_2),
    ('5.3 paper T=10', rec_5_3_paper),
    ('5.3 edp T=10', rec_5_3_edp),
    ('5.4 paper', rec_5_4),
    ('5.5 paper', rec_5_5),
]

summary_rows = []
print(f'{"example":<22s} {"n_seg":>6s} {"inner":>7s} {"PDE/seg":>9s} '
      f'{"IoU mean":>9s} {"IoU max":>8s} {"IoU last20":>10s} {"runtime":>9s}')
print('-' * 91)
for label, rec in records:
    n_inner = rec['n_inner_per_segment']
    iou = rec['iou_history']
    inner_mean = float(n_inner.mean())
    pde_per_seg = 2.0 * inner_mean + 3.0
    last20 = float(iou[-20:].mean()) if iou.size >= 20 else float(iou.mean())
    row = {
        'example': label,
        'n_seg': int(iou.shape[0]),
        'inner_mean': inner_mean,
        'pde_per_seg_est': pde_per_seg,
        'iou_mean': float(iou.mean()),
        'iou_max': float(iou.max()),
        'iou_last20': last20,
        'runtime_seconds': float(rec['runtime_seconds']),
    }
    summary_rows.append(row)
    print(f'{label:<22s} {row["n_seg"]:>6d} {inner_mean:>7.2f} {pde_per_seg:>9.2f} '
          f'{row["iou_mean"]:>9.3f} {row["iou_max"]:>8.3f} {last20:>10.3f} '
          f'{row["runtime_seconds"]:>8.1f}s')


## 10. Summary and Discussion

### Phase 5 Deliverables

| Deliverable | Evidence |
|---|---|
| Three-mesh parabolic flow | Section 1 constructs `ThFine / Th / ThCoarse`; live runs call `run_idsm_parabolic(..., solve_mesh=solve_mesh)`. |
| Algorithm 4.1 segment loop | Sections 4-8 call `run_idsm_parabolic` for all five paper examples. |
| Noise robustness | Section 4.1 compares Example 5.1 at 5% and 10% noise. |
| Nonlinear recovery | Section 6 runs the dedicated U-recovery branch for Example 5.3. |
| Reproducible figures | Each plot is saved under `../figures/05_*.png`. |
| Live numerical summary | Section 9 prints the table from `summary_rows`, not from fixed markdown values. |

### Notes on Special Cases

- **Example 5.3 (nonlinear U-recovery)**: the original `parabolic_Nonlinear.edp` default `totalTime=0.51` is a short run. The long-time run in this notebook sets `total_time=10.0` to match the paper's qualitative trajectory discussion.
- **Example 5.4 (potential fading)**: conductivity is constant, so IoU is computed on the V channel rather than the trivial conductivity channel.
- **paper vs edp parameters**: `paper_cfg_example_5_*` keeps the same FreeFEM numerical defaults as `edp_cfg_example_5_*`; the distinction is the paper-style noise level supplied by the notebook or driver. Use `scripts/run_all_examples.py --mesh-mode edp` for the strict FreeFEM three-mesh run.

### Comparison with Elliptic IDSM

| Aspect | Elliptic IDSM (NB03) | Parabolic IDSM (NB05) |
|---|---|---|
| Unknown | static coefficient | moving coefficient field over time segments |
| Forward model | elliptic Neumann solve | Crank-Nicolson parabolic solve |
| Data | boundary Cauchy data at one state | lateral boundary data over time |
| Kernel update | one low-rank sequence | segment-wise low-rank update with forgetting |
| Output | one reconstructed inclusion map | time-indexed reconstruction history |

The essential new difficulty is temporal accumulation: a useful correction in one segment can become stale in the next because the inclusion has moved. Algorithm 4.1 therefore combines local-in-time adjoint information, low-rank correction, and inter-segment damping.